# 🎀 林星蕾 LoRA 訓練營 v2 — 全自動版
## 設備：T4 GPU · SDXL · 1024×1024 · 118張圖片
---
**上傳 zip → 一鍵訓練 → 自動下載，全程免手動！**
---
### 使用方法
1. 按 `Ctrl+F9` 或「執行階段 → 全部執行」
2. 上傳 `lora_training_data.zip`（自動跳出）
3. （可選）輸入 HuggingFace Token 加速下載
4. 等待訓練完成（約 1-2 小時），自動下載 LoRA
---
### 訓練參數
| 項目 | 數值 |
|------|------|
| 基底模型 | SDXL 1.0 |
| 解析度 | 1024×1024 |
| LoRA dim/alpha | 32 / 16 |
| 訓練步數 | 1200 (~10 epochs) |
| 學習率 | 1e-4 (cosine) |
| 觸發詞 | `linxinglei` |
| 優化器 | AdamW8bit |

In [ ]:
# === 1. 安裝依賴 ===
!pip install -q --upgrade pip
!pip install -q accelerate xformers --index-url https://download.pytorch.org/whl/cu121
!pip install -q bitsandbytes transformers ftfy tensorboard safetensors
!pip install -q lion-pytorch optimi opencv-python-headless huggingface_hub
print('✅ 依賴安裝完成')

In [ ]:
# === 2. 設定 Accelerate ===
from accelerate.utils import write_basic_config
write_basic_config(
    mixed_precision='bf16',
    use_cpu=False,
    num_processes=1,
    num_machines=1,
    machine_rank=0
)
print('✅ Accelerate 設定完成 (bf16)')

In [ ]:
# === 3. 下載 lora_training_data.zip (自動從 GitHub 下載) ===
import urllib.request, os

ZIP_URL = 'https://github.com/vito0527opencode/lora-training/raw/main/lora_training_data.zip'
ZIP_PATH = '/content/lora_training_data.zip'

if not os.path.exists(ZIP_PATH):
    print('📦 自動下載訓練素材...')
    urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
    print(f'✅ 下載完成: {os.path.getsize(ZIP_PATH)//1024//1024} MB')
else:
    print(f'✅ zip 已存在: {ZIP_PATH}')

# 驗證
assert os.path.exists(ZIP_PATH), 'zip 下載失敗！'
print(f'📦 zip 大小: {os.path.getsize(ZIP_PATH)//1024//1024} MB')


In [ ]:
# === 4. 解壓縮 + 產生個別 caption 檔案 ===
import zipfile, glob, os

zips = glob.glob('/content/*.zip')
assert zips, '找不到 zip 檔！請先上傳。'

with zipfile.ZipFile(zips[0], 'r') as z:
    z.extractall('/content/training_data')

# 找 processed/ 目錄
processed_dir = None
for root, dirs, files in os.walk('/content/training_data'):
    if 'captions.txt' in files:
        processed_dir = root
        break

if not processed_dir:
    # 如果沒有 processed 子目錄，可能直接解壓在 training_data
    processed_dir = '/content/training_data/processed'
    if not os.path.exists(processed_dir):
        processed_dir = '/content/training_data'

# 讀取 captions.txt，產生每個圖片的 .txt caption
caption_file = os.path.join(processed_dir, 'captions.txt')
if not os.path.exists(caption_file):
    # 可能在 training_data 根目錄
    caption_file = '/content/training_data/captions.txt'

assert os.path.exists(caption_file), f'找不到 captions.txt! 檢查過: {processed_dir}'

TRIGGER = 'linxinglei'
img_count = 0
with open(caption_file, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line or '|' not in line:
            continue
        img_name, caption = line.split('|', 1)
        img_path = os.path.join(processed_dir, img_name)
        txt_path = os.path.join(processed_dir, img_name.rsplit('.', 1)[0] + '.txt')
        if os.path.exists(img_path):
            with open(txt_path, 'w', encoding='utf-8') as cf:
                cf.write(f'{TRIGGER}, {caption}')
            img_count += 1

print(f'✅ 解壓完成，訓練目錄: {processed_dir}')
print(f'✅ 產生 {img_count} 個 caption 檔案（觸發詞: {TRIGGER}）')

# 列出幾張預覽
import random
samples = random.sample([f for f in os.listdir(processed_dir) if f.endswith('.png')], min(5, img_count))
for s in sorted(samples):
    txt = s.rsplit('.', 1)[0] + '.txt'
    txt_path = os.path.join(processed_dir, txt)
    cap = open(txt_path).read().strip() if os.path.exists(txt_path) else '(無 caption)'
    print(f'  📸 {s} → {cap}')

In [ ]:
# === 5. 克隆 Kohya SS ===
!git clone -q https://github.com/bmaltais/kohya_ss.git /content/kohya_ss
%cd /content/kohya_ss
!pip install -q -r requirements.txt 2>&1 | tail -3

# 確保 accelerate 設定正確
from accelerate.utils import write_basic_config
write_basic_config(mixed_precision='bf16', use_cpu=False)

# 找訓練腳本
import os
for root, dirs, files in os.walk('/content/kohya_ss'):
    if 'sdxl_train_network.py' in files:
        TRAIN_SCRIPT = os.path.join(root, 'sdxl_train_network.py')
        break
else:
    raise FileNotFoundError('找不到 sdxl_train_network.py！')

print(f'✅ Kohya SS 安裝完成')
print(f'📝 訓練腳本: {TRAIN_SCRIPT}')

In [ ]:
# === 6. (可選) 設定 HuggingFace Token 加速下載 SDXL ===
# 如果已有 token 可填入加速下載，沒有的話會自動下載（較慢）

HF_TOKEN = ''  # ← 貼上你的 HuggingFace READ token （選填）

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print('✅ HuggingFace 已登入')
else:
    print('ℹ️  未設定 HF token，將自動下載（不需登入也可下載公開模型）')

In [ ]:
# === 7. 🚀 開始訓練 ===
# 找到 processed_dir (from cell 4)
import os
processed_dir = None
for root, dirs, files in os.walk('/content/training_data'):
    if 'captions.txt' in files:
        processed_dir = root
        break
if not processed_dir:
    processed_dir = '/content/training_data/processed'
    if not os.path.exists(processed_dir):
        processed_dir = '/content/training_data'

print(f'📂 訓練資料: {processed_dir}')
print(f'📂 輸出目錄: /content/output')
print()
print('🔥 開始訓練... (約 1-2 小時)')
!accelerate launch --num_cpu_threads_per_process 2 "{TRAIN_SCRIPT}" \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --train_data_dir="{processed_dir}" \
  --output_dir="/content/output" \
  --output_name="linxinglei_lora_v1" \
  --resolution="1024,1024" \
  --train_batch_size=1 \
  --network_module="networks.lora" \
  --network_dim=32 \
  --network_alpha=16 \
  --max_train_steps=1200 \
  --learning_rate=1e-4 \
  --optimizer_type="AdamW8bit" \
  --mixed_precision="bf16" \
  --save_every_n_epochs=2 \
  --save_model_as="safetensors" \
  --caption_extension=".txt" \
  --cache_latents \
  --lr_scheduler="cosine" \
  --lr_warmup_steps=50 \
  --enable_bucket \
  --min_bucket_reso=256 \
  --max_bucket_reso=2048 \
  --keep_tokens=0 \
  --xformers \
  --no_half_vae

print()
print('='*50)
print('🎉 訓練完成！')print('='*50)

In [ ]:
# === 8. 下載 LoRA ===
import glob, os
from datetime import datetime

print('📦 尋找 LoRA 檔案...')

safetensors_files = glob.glob('/content/output/**/*.safetensors', recursive=True)

if not safetensors_files:
    # 嘗試其他可能路徑
    for alt in ['/content/kohya_ss/output', '/content/drive/MyDrive']:
        safetensors_files = glob.glob(f'{alt}/**/*.safetensors', recursive=True)
        if safetensors_files:
            break

if safetensors_files:
    latest = max(safetensors_files, key=os.path.getmtime)
    size_mb = os.path.getsize(latest) / 1024 / 1024
    mtime = datetime.fromtimestamp(os.path.getmtime(latest))
    print(f'📦 找到 LoRA: {latest}')
    print(f'📏 大小: {size_mb:.1f} MB')
    print(f'🕐 時間: {mtime}')
    print()
    print('⬇️  開始下載...')
    from google.colab import files
    files.download(latest)
else:
    print('⚠️ 找不到 .safetensors 檔案')
    print('檢查 /content/output/ 目錄：')
    !find /content/output -type f 2>/dev/null | head -20

In [ ]:
# === 9. (額外) 備份到 Google Drive ===
from google.colab import drive
import shutil, os, glob

try:
    drive.mount('/content/drive')
    print('✅ Google Drive 已掛載')
    
    output = '/content/output'
    backup_dir = '/content/drive/MyDrive/LoRA/linxinglei'
    os.makedirs(backup_dir, exist_ok=True)
    
    safetensors_files = glob.glob(f'{output}/**/*.safetensors', recursive=True)
    for f in safetensors_files:
        dest = os.path.join(backup_dir, os.path.basename(f))
        shutil.copy2(f, dest)
        print(f'📁 已備份: {dest}')
    print('✅ 備份完成')
except Exception as e:
    print(f'ℹ️ 跳過 Drive 備份: {e}')

---
## 🎉 完成！

LoRA 已經自動下載到你的電腦了～

### 使用方式
```
在 ComfyUI / A1111 / SD.Next 載入 linxinglei_lora_v1.safetensors
使用觸發詞: linxinglei
權重建議: 0.6 ~ 0.8
```

### 問題排查
- **找不到 GPU**：確認 Colab 執行階段 → 變更執行階段類型 → T4 GPU
- **OOM（記憶體不足）**：重啟執行階段再試
- **下載失敗**：在左側檔案面板手動下載 `/content/output/` 中的 .safetensors